# ShopSegments — Day 1: Cleaning, RFM Features, EDA

**Project:** Customer Segmentation & Product Recommender (DawoodTech ML — Week 3)
**Dataset:** Online Retail II (UCI) — UK online gift retailer, Dec 2009 – Dec 2011

## Problem (business ki zaban me)
Online stores sabko same promotion bhejte hain aur paisa waste hota hai. ShopSegments customers ko unke **behaviour** ke hisaab se groups me baantta hai aur har customer ko sahi products recommend karta hai.

## Is notebook (Day 1) ka goal
Transactions ki table (har row = ek product ki ek khareed) ko **customers ki table** (har row = ek customer) me badalna:

```
transactions (1M+ rows)  ->  clean transactions  ->  RFM table (1 row per customer)
```

## Is notebook me kya hai
1. Setup aur data load (dono sheets)
2. Data audit (cleaning se **pehle** sirf dekhna)
3. Cleaning (har step ka row-count log ke saath)
4. Non-product StockCodes
5. RFM features banana + validation
6. EDA (plots + insights)
7. Day 2 ke liye preview (log transform)
8. Files save + reusable script `src/data_prep.py`

> Har section ke upar markdown me **kyun** likha hai. Run karne se pehle padho, run karne ke baad output dekho, phir "Insight" wale cells me apne words me likho. Yahi tumhari README aur mentor demo ka material banega.

## 1. Setup

Libraries import karte hain aur folders ka structure bana lete hain. Notebook project root me ho ya `notebooks/` folder me, paths dono me kaam karenge.

Agar koi library missing ho: `pip install pandas numpy matplotlib seaborn pyarrow openpyxl`

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)

# Project root: agar notebook notebooks/ ke andar hai to ek level upar
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = BASE / "data" / "raw"
PROC_DIR = BASE / "data" / "processed"
SRC_DIR = BASE / "src"
for d in (RAW_DIR, PROC_DIR, SRC_DIR, BASE / "models", BASE / "notebooks", BASE / "app"):
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE)

## 2. Data load (dono sheets)

Excel file me **2 sheets** hain:
- `Year 2009-2010` (Dec 2009 – Dec 2010)
- `Year 2010-2011` (Dec 2010 – Dec 2011)

`pd.read_excel(file)` bina `sheet_name` ke **sirf pehli sheet** padhta hai, isliye `sheet_name=None` dete hain (sab sheets ka dictionary milta hai) aur `concat` se jod dete hain.

**Excel padhna slow hai (~1-2 min)**, isliye pehli baar padhke `parquet` me save kar dete hain. Agli baar parquet se load hoga, jo seconds me ho jata hai.

**File kahan rakhni hai:** `data/raw/online_retail_II.xlsx` (ya notebook ke folder me).

In [ ]:
RAW_PARQUET = PROC_DIR / "raw_combined.parquet"

def find_excel():
    for p in (RAW_DIR / "online_retail_II.xlsx", BASE / "online_retail_II.xlsx", Path("online_retail_II.xlsx")):
        if p.exists():
            return p
    raise FileNotFoundError("online_retail_II.xlsx nahi mili. Isko data/raw/ me rakho.")

if RAW_PARQUET.exists():
    df_raw = pd.read_parquet(RAW_PARQUET)
    print("Parquet se load hua (fast).")
else:
    sheets = pd.read_excel(find_excel(), sheet_name=None)      # {sheet_name: DataFrame}
    for name, s in sheets.items():
        print(f"{name:20s} {s.shape}")
    df_raw = pd.concat(sheets.values(), ignore_index=True)
    # Invoice/StockCode me numbers aur text mix hote hain (jaise 489434 aur C489449),
    # parquet ko ek hi type chahiye, isliye pehle text bana do
    df_raw["Invoice"] = df_raw["Invoice"].astype(str)
    df_raw["StockCode"] = df_raw["StockCode"].astype(str)
    # Description me bhi kahin kahin number aa jata hai; NaN ko NaN hi rehne do
    df_raw["Description"] = df_raw["Description"].where(df_raw["Description"].isna(), df_raw["Description"].astype(str))
    df_raw.to_parquet(RAW_PARQUET)
    print("Excel se load hua aur parquet me save kar diya.")

print("Combined shape:", df_raw.shape)      # expected (1067371, 8)

**Check:** shape `(1067371, 8)` aani chahiye (525,461 + 541,910). Agar kam aaye to ek sheet poori load nahi hui.

### Columns ka matlab (is file me asli naam)
| Column | Matlab | Kahan use hoga |
|---|---|---|
| `Invoice` | Invoice number. `C` se shuru ho to cancellation | Frequency, cancellations |
| `StockCode` | Product code (kuch codes me letter bhi, jaise `85123A`) | Recommender ka item ID |
| `Description` | Product ka naam | Dashboard me dikhane ke liye |
| `Quantity` | Kitne units | Monetary |
| `InvoiceDate` | Khareed ka date-time | Recency, time-based split |
| `Price` | Per unit price (£) | Monetary |
| `Customer ID` | Customer ka 5-digit ID | RFM ka groupby key, recommender ka user |
| `Country` | Customer ka mulk | EDA / segment profile |

> Note: UCI page pe naam `InvoiceNo`, `UnitPrice`, `CustomerID` likhe hote hain, lekin **is file me** `Invoice`, `Price`, `Customer ID` (space ke saath) hain.

## 3. Data audit (pehle sirf dekho, kuch drop mat karo)

Cleaning se pehle samajhna zaroori hai ki data me **kya kharabi hai** aur **kitni**. Har finding README ke "Limitations" section me kaam aayegi.

In [ ]:
display(df_raw.head())
print()
df_raw.info()

In [ ]:
print("Date range :", df_raw["InvoiceDate"].min(), "->", df_raw["InvoiceDate"].max())
print("Duplicates :", df_raw.duplicated().sum(), "(exact same row)")

print("\nMissing values:")
miss = df_raw.isnull().sum().to_frame("missing")
miss["percent"] = miss["missing"] / len(df_raw) * 100
display(miss)

**Kya dekhna hai:**
- `Customer ID` me ~22% missing hai. Customer ke bina RFM nahi ban sakta, isliye ye rows drop hongi.
- Duplicates (~34k): dono sheets me **1-9 Dec 2010 overlap** karte hain, isliye exact repeat rows hain.

In [ ]:
inv = df_raw["Invoice"].astype(str)
is_cancel = inv.str.startswith("C")

print("Cancelled invoices (C prefix)     :", is_cancel.sum())
print("Quantity <= 0                     :", (df_raw["Quantity"] <= 0).sum())
print("Price <= 0                        :", (df_raw["Price"] <= 0).sum())
print("Cancelled AND negative quantity   :", (is_cancel & (df_raw["Quantity"] < 0)).sum())
print("Negative qty but NOT cancelled    :", (~is_cancel & (df_raw["Quantity"] < 0)).sum())
print("Cancelled but positive quantity   :", (is_cancel & (df_raw["Quantity"] > 0)).sum())

**Samajh:**
- `C` wali invoices returns/cancellations hain. Inme Quantity negative hoti hai.
- Negative quantity bina `C` ke = stock adjustment / damaged goods wali entries (data errors jaisi).
- Agar ye rows RFM me rakhein to customer ka Monetary galat kam ho jayega.

In [ ]:
country_share = (df_raw["Country"].value_counts(normalize=True) * 100).head(10)
display(country_share.to_frame("% of rows"))

df_raw["Country"].value_counts().head(10).sort_values().plot(kind="barh", figsize=(8, 4), title="Top 10 countries by rows")
plt.show()

**Insight (apne words me likho):** UK dominate karta hai (~92% rows, ~83% revenue). Matlab segmentation mostly UK customers ki hogi; ye README ki limitation hai.

_Tumhara insight:_ ...

## 4. Cleaning (har step ke baad row-count log)

Order matter karta hai, aur har step ka log isliye rakhte hain ki mentor poochhe "kitna data hataya aur kyun?", to exact jawab ho.

| Step | Kyun |
|---|---|
| Exact duplicates drop | Overlap wali / repeat rows, ek hi transaction do baar count hoti |
| Null date / quantity / price drop | Inke bina row ka koi matlab nahi (safety step) |
| Missing `Customer ID` drop | Customer-level analysis ke liye ID chahiye |
| Cancelled invoices drop | Returns RFM ko kharab karte hain |
| `Quantity <= 0` ya `Price <= 0` drop | Data errors / adjustments / free items |
| `LineTotal = Quantity x Price` | Dataset me total column nahi hai, khud banana hai |
| `Customer ID` ko int me | NaN ki wajah se float tha (`12346.0`) |

In [ ]:
df = df_raw.copy()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

log = [("0. start (combined raw)", len(df))]

df = df.drop_duplicates()
log.append(("1. drop exact duplicates", len(df)))

df = df.dropna(subset=["InvoiceDate", "Quantity", "Price"])
log.append(("2. drop null date/qty/price", len(df)))

df = df.dropna(subset=["Customer ID"])
log.append(("3. drop missing Customer ID", len(df)))

df = df[~df["Invoice"].str.startswith("C")]
log.append(("4. drop cancelled invoices", len(df)))

df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
log.append(("5. drop qty<=0 or price<=0", len(df)))

df["Customer ID"] = df["Customer ID"].astype(int)
df["LineTotal"] = df["Quantity"] * df["Price"]
df = df.reset_index(drop=True)

cleaning_log = pd.DataFrame(log, columns=["step", "rows_after"])
cleaning_log["rows_removed"] = (-cleaning_log["rows_after"].diff()).fillna(0).astype(int)
cleaning_log["% of raw removed"] = cleaning_log["rows_removed"] / log[0][1] * 100
display(cleaning_log)
print(f"\nFinal clean rows: {len(df):,}  ({len(df)/log[0][1]*100:.1f}% of raw bacha)")

**Expected numbers** (is dataset ke liye): duplicates 34,335 -> 1,033,036 rows, missing ID drop -> 797,885, cancelled drop -> 779,495, qty/price drop -> **779,425**.

Ye table README ke "Data cleaning" section me daal dena.

**Limitation (honestly likhna):** ~23% rows `Customer ID` ke bina thi, wo hata di. Ho sakta hai wo guest checkouts hon, isliye hamare segments sirf **registered customers** ke hain.

In [ ]:
print("Missing Description after cleaning:", df["Description"].isna().sum())
print("Unique customers :", df["Customer ID"].nunique())
print("Unique invoices  :", df["Invoice"].nunique())
print("Unique products  :", df["StockCode"].nunique())
print("Total sales (GBP):", round(df["LineTotal"].sum(), 2))
print("Date range       :", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())

## 5. Non-product StockCodes

Kuch `StockCode` asli products nahi hain: `POST` (postage), `D` (discount), `M` (manual), `BANK CHARGES`, `TEST001`, `ADJUST` etc.

Asli product codes **5 digits se shuru** hote hain (`85123A` me 5 digit + letter variant). Isliye regex `^\d{5}` se product / non-product alag karte hain.

**Decision:**
- **RFM me rakhenge** (postage bhi customer ka asli kharch hai, aur impact chhota hai, neeche number dekho).
- **Recommender me hatayenge** (Day 4), kyunki "Postage" ko recommend karna bekaar hai.
- Isliye abhi sirf ek flag column `is_product` banate hain.

In [ ]:
df["is_product"] = df["StockCode"].str.match(r"^\d{5}")

non_prod = df[~df["is_product"]]
display(non_prod["StockCode"].value_counts().head(15).to_frame("rows"))

share = non_prod["LineTotal"].sum() / df["LineTotal"].sum() * 100
print(f"\nNon-product rows : {len(non_prod):,} ({len(non_prod)/len(df)*100:.2f}% of rows)")
print(f"Non-product spend: {share:.2f}% of total sales")

## 6. RFM features

**RFM** customer ke behaviour ke 3 numbers hain:

| Feature | Matlab | Achha kya hai |
|---|---|---|
| **Recency (R)** | Last khareed ko kitne **din** hue | **Chhota** (haal hi me aaya) |
| **Frequency (F)** | Kitni **alag invoices** (orders) | **Bada** |
| **Monetary (M)** | Total kitna kharch kiya (£) | **Bada** |

### Do zaroori baatein
1. **Snapshot date:** Recency "aaj" (2026) se nahi, dataset ki **last date + 1 din** se nikalte hain. Warna sab customers "bahut purane" dikhte. `+1` isliye ki last din khareedne wale ki Recency 0 nahi, 1 aaye. Last khareed ki date ko bhi `normalize()` karte hain (time hata ke sirf date), warna 12:50 jaisa time din ginte waqt gir jata hai aur Recency 0 aa jati hai.
2. **Frequency = `nunique(Invoice)`**, rows ki ginti nahi. Ek invoice me 20 products ho sakte hain, wo **1 order** hai.

In [ ]:
snapshot_date = df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

g = df.groupby("Customer ID")
rfm = pd.DataFrame({
    "Recency":   (snapshot_date - g["InvoiceDate"].max().dt.normalize()).dt.days,   # normalize: sirf date, time nahi
    "Frequency": g["Invoice"].nunique(),
    "Monetary":  g["LineTotal"].sum(),
}).reset_index()

features = ["Recency", "Frequency", "Monetary"]
print("RFM shape:", rfm.shape)           # ~5,878 customers
display(rfm.head(10))
display(rfm[features].describe())

### Validation (sanity checks)
`assert` ek safety net hai: condition galat ho to code ruk jata hai aur pata chal jata hai ki bug kahan hai, bajaye iske ki galat data chup chaap aage chala jaye.

In [ ]:
assert rfm["Customer ID"].is_unique,        "Ek customer ki ek se zyada rows!"
assert rfm.isna().sum().sum() == 0,         "RFM me null aa gaye"
assert (rfm["Recency"] >= 1).all(),         "Recency 1 se kam"
assert (rfm["Frequency"] >= 1).all(),       "Frequency 1 se kam"
assert (rfm["Monetary"] > 0).all(),         "Monetary positive nahi"
assert np.isclose(rfm["Monetary"].sum(), df["LineTotal"].sum()), "Total sales match nahi karti"
print("Saare checks pass.")

### Manual verification (5 random customers)
Code chal gaya matlab sahi chala, ye zaroori nahi. Yahan 5 customers ke numbers raw transactions se **dobara alag tarike se** nikalke RFM table se compare karte hain.

In [ ]:
sample_ids = rfm["Customer ID"].sample(5, random_state=42)
rows = []
for cid in sample_ids:
    c = df[df["Customer ID"] == cid]
    manual_R = (snapshot_date - c["InvoiceDate"].max().normalize()).days
    manual_F = c["Invoice"].nunique()
    manual_M = c["LineTotal"].sum()
    r = rfm[rfm["Customer ID"] == cid].iloc[0]
    ok = (manual_R == r["Recency"]) and (manual_F == r["Frequency"]) and np.isclose(manual_M, r["Monetary"])
    rows.append([cid, manual_R, r["Recency"], manual_F, r["Frequency"], round(manual_M, 2), round(r["Monetary"], 2), ok])

check = pd.DataFrame(rows, columns=["Customer ID", "R manual", "R table", "F manual", "F table", "M manual", "M table", "match"])
display(check)
assert check["match"].all()

## 7. EDA (Exploratory Data Analysis)

Plots banane ka maqsad: customers ke behaviour ko **samajhna**, aur Day 2 ke decisions ke liye evidence jama karna. Har plot ke neeche insight likhna hai.

### 7.1 R, F, M ka distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col in zip(axes, features):
    sns.histplot(rfm[col], bins=50, kde=True, ax=ax)
    ax.set_title(f"{col} distribution")
    ax.set_ylabel("Number of customers")
plt.tight_layout()
plt.show()

skew = rfm[features].skew().round(2)
display(skew.to_frame("skewness"))

**Skewness kya hai:** distribution ki "dum" kis taraf lambi hai. 0 ke paas = symmetric. **1 se bada = heavily right-skewed** (zyada tar customers kam kharch karte hain, kuch bahut zyada).

**Insight:** teeno features right-skewed hain: Monetary (~25) aur Frequency (~12.6) bahut zyada, Recency (~0.9) halki. Agar bina transform ke K-Means chalayein to bade spenders alag ho jayenge aur baaki sab ek blob me aa jayenge. Isliye Day 2 me `log1p` lagega.

_Tumhara insight:_ ...

### 7.2 Outliers (boxplots) aur correlation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, features):
    sns.boxplot(y=rfm[col], ax=ax)
    ax.set_title(f"{col} boxplot")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
sns.heatmap(rfm[features].corr(), annot=True, cmap="Blues", vmin=-1, vmax=1)
plt.title("Correlation (raw RFM)")
plt.show()

**Insight:** boxplot me Frequency aur Monetary ke bahut saare outliers hain (wholesale buyers jaise bade customers). Correlation heatmap me Frequency aur Monetary positively correlated hain (jo zyada baar aata hai wo zyada kharchta hai), aur Recency in dono se negatively correlated hai (jo haal hi me aaya wo zyada active hota hai). Exact values heatmap me dekho.

_Tumhara insight:_ ...

### 7.3 Time trend: monthly revenue aur orders

In [ ]:
monthly = (
    df.assign(month=df["InvoiceDate"].dt.to_period("M"))
      .groupby("month")
      .agg(revenue=("LineTotal", "sum"), orders=("Invoice", "nunique"), customers=("Customer ID", "nunique"))
)
monthly.index = monthly.index.to_timestamp()
monthly.plot(subplots=True, figsize=(11, 8), marker="o", title=["Monthly revenue (GBP)", "Monthly orders", "Active customers"], legend=False)
plt.tight_layout()
plt.show()

**Insight:** Sep-Nov me sales ka bada uthaal (Christmas gifting season), aur Dec me kami. Ye retailer gift items bechta hai, isliye seasonality strong hai. Last month (Dec 2011) adha hai (data 9 Dec tak), isliye wahan girawat real nahi.

_Tumhara insight:_ ...

### 7.4 Top countries aur top products

In [ ]:
country_rev = df.groupby("Country")["LineTotal"].sum().sort_values(ascending=False)
display((country_rev.head(10) / country_rev.sum() * 100).round(2).to_frame("% of revenue"))

prod = df[df["is_product"]]
top_qty = prod.groupby("Description")["Quantity"].sum().nlargest(10)
top_rev = prod.groupby("Description")["LineTotal"].sum().nlargest(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
top_qty.sort_values().plot(kind="barh", ax=axes[0], title="Top 10 products by quantity")
top_rev.sort_values().plot(kind="barh", ax=axes[1], title="Top 10 products by revenue (GBP)")
plt.tight_layout()
plt.show()

**Insight:** quantity me jo products upar hain (sasti cheezein, bulk me bikti hain) aur revenue me jo upar hain, dono lists alag hoti hain. Popularity-based recommender (Day 4) isi se banega.

_Tumhara insight:_ ...

### 7.5 One-time customers aur Pareto (80/20 rule)

In [ ]:
one_time = (rfm["Frequency"] == 1).mean() * 100
print(f"Sirf 1 order karne wale customers: {one_time:.1f}%")
print(f"Median orders per customer       : {rfm['Frequency'].median():.0f}")

# Pareto: top 10% customers ka revenue share
sorted_m = rfm["Monetary"].sort_values(ascending=False).reset_index(drop=True)
top10 = sorted_m.head(int(len(sorted_m) * 0.10)).sum() / sorted_m.sum() * 100
top20 = sorted_m.head(int(len(sorted_m) * 0.20)).sum() / sorted_m.sum() * 100
print(f"Top 10% customers ka revenue share: {top10:.1f}%")
print(f"Top 20% customers ka revenue share: {top20:.1f}%")

cum = sorted_m.cumsum() / sorted_m.sum() * 100
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(cum) + 1) / len(cum) * 100, cum)
plt.axvline(20, ls="--", c="gray"); plt.axhline(top20, ls="--", c="gray")
plt.xlabel("Top X% customers"); plt.ylabel("Cumulative % of revenue"); plt.title("Pareto curve")
plt.show()

**Insight:** top 10% customers ~64% aur top 20% ~77% revenue dete hain (Pareto), aur ~28% customers ne sirf 1 order kiya. Isi wajah se segmentation useful hai: "Champions" ko alag treat karna marketing budget bachata hai.

**Recommender ke liye important:** one-time customers ki history chhoti hoti hai, unke liye personalised recommendation weak hogi. Isliye cold-start fallback (popular items) zaroori hai.

_Tumhara insight:_ ...

## 8. Day 2 preview: log transform ka asar

Day 2 me hum K-Means chalayenge. K-Means **distance** pe chalta hai, isliye skewed data pe bade values sab kuch dominate kar lete hain. `np.log1p(x) = log(1 + x)` bade values ko dabata hai aur chhote values ko failata hai (`1+` isliye ki 0 pe error na aaye).

Abhi sirf **dekho** ki skew kitni kam hoti hai. Scaling aur clustering kal.

In [ ]:
rfm_log = rfm[features].apply(np.log1p)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col in zip(axes, features):
    sns.histplot(rfm_log[col], bins=40, kde=True, ax=ax, color="seagreen")
    ax.set_title(f"log1p({col})")
plt.tight_layout()
plt.show()

compare = pd.DataFrame({"skew before": rfm[features].skew(), "skew after log1p": rfm_log.skew()}).round(2)
display(compare)

**Insight:** log ke baad Monetary ki skew ~25 se ~0.3 aur Frequency ki ~12.6 se ~1.0 reh jati hai, yaani distributions kaafi symmetric ho gaye. Recency pehle halki right-skewed thi (0.89), log ke baad halki **left**-skewed (-0.45) ho jati hai, ye chhoti baat hai aur clustering ko kharab nahi karti.

_Tumhara insight:_ ...

## 9. Outputs save karna

Day 2 se in files se kaam shuru hoga (Excel dobara kabhi nahi padhni):
- `clean_transactions.parquet`: clean transaction rows (recommender ke liye bhi)
- `rfm.parquet`: customer-level RFM table
- `products.parquet`: har StockCode ka ek description (dashboard me product naam dikhane ke liye)

In [ ]:
df.to_parquet(PROC_DIR / "clean_transactions.parquet", index=False)
rfm.to_parquet(PROC_DIR / "rfm.parquet", index=False)

# StockCode -> sabse common description (same code ke kai naam ho sakte hain)
products = (
    df.groupby("StockCode")["Description"]
      .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN")
      .reset_index()
)
products.to_parquet(PROC_DIR / "products.parquet", index=False)

for f in ("clean_transactions", "rfm", "products"):
    p = PROC_DIR / f"{f}.parquet"
    print(f"{p.name:28s} {p.stat().st_size/1e6:6.1f} MB")

## 10. Reusable script: `src/data_prep.py`

Task me likha hai: **"reusable Python scripts (not only notebooks)"**. Notebook me prototype kiya, ab wahi logic ek `.py` file me daal dete hain, jise Day 5 me Streamlit app import karega.

Neeche wala cell file `src/data_prep.py` bana dega.

In [ ]:
script = r'''"""Data preparation for ShopSegments: load, clean, and build RFM features."""
from pathlib import Path

import pandas as pd


def load_data(excel_path):
    """Read both sheets of Online Retail II and combine them."""
    sheets = pd.read_excel(excel_path, sheet_name=None)
    return pd.concat(sheets.values(), ignore_index=True)


def clean_transactions(df_raw):
    """Return clean purchase rows with LineTotal and an is_product flag."""
    df = df_raw.copy()
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
    df["Invoice"] = df["Invoice"].astype(str)
    df["StockCode"] = df["StockCode"].astype(str)

    df = df.drop_duplicates()
    df = df.dropna(subset=["InvoiceDate", "Quantity", "Price", "Customer ID"])
    df = df[~df["Invoice"].str.startswith("C")]            # cancellations
    df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

    df["Customer ID"] = df["Customer ID"].astype(int)
    df["LineTotal"] = df["Quantity"] * df["Price"]
    df["is_product"] = df["StockCode"].str.match(r"^\d{5}")
    return df.reset_index(drop=True)


def build_rfm(df, snapshot_date=None):
    """One row per customer: Recency (days), Frequency (orders), Monetary (GBP)."""
    if snapshot_date is None:
        snapshot_date = df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
    g = df.groupby("Customer ID")
    rfm = pd.DataFrame({
        "Recency": (snapshot_date - g["InvoiceDate"].max().dt.normalize()).dt.days,
        "Frequency": g["Invoice"].nunique(),
        "Monetary": g["LineTotal"].sum(),
    }).reset_index()
    return rfm


if __name__ == "__main__":
    base = Path(__file__).resolve().parents[1]
    raw = load_data(base / "data" / "raw" / "online_retail_II.xlsx")
    clean = clean_transactions(raw)
    rfm = build_rfm(clean)
    (base / "data" / "processed").mkdir(parents=True, exist_ok=True)
    clean.to_parquet(base / "data" / "processed" / "clean_transactions.parquet", index=False)
    rfm.to_parquet(base / "data" / "processed" / "rfm.parquet", index=False)
    print("clean:", clean.shape, "| rfm:", rfm.shape)
'''

(SRC_DIR / "data_prep.py").write_text(script, encoding="utf-8")
print("Saved:", SRC_DIR / "data_prep.py")

**Script ko verify karo** (notebook ka result aur script ka result same hona chahiye):

In [ ]:
import importlib, sys
sys.path.insert(0, str(SRC_DIR))
import data_prep
importlib.reload(data_prep)

clean_check = data_prep.clean_transactions(df_raw)
rfm_check = data_prep.build_rfm(clean_check)

assert clean_check.shape[0] == df.shape[0], "clean rows match nahi"
assert np.allclose(rfm_check.sort_values("Customer ID")[features].values,
                   rfm.sort_values("Customer ID")[features].values), "RFM match nahi"
print("Script aur notebook ka output bilkul same hai.")

## 11. Day 1 summary (README ke liye)

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Raw rows (both sheets)", "Clean transaction rows", "Rows removed (%)",
        "Unique customers (RFM rows)", "Unique products", "Date range",
        "Snapshot date", "Customers with 1 order (%)", "Top 10% customers' revenue share (%)",
        "Non-product spend share (%)",
    ],
    "value": [
        f"{len(df_raw):,}", f"{len(df):,}", f"{(1 - len(df)/len(df_raw))*100:.1f}",
        f"{len(rfm):,}", f"{df['StockCode'].nunique():,}",
        f"{df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}",
        str(snapshot_date.date()), f"{one_time:.1f}", f"{top10:.1f}", f"{share:.2f}",
    ],
})
display(summary)

### Day 1 checklist
- [x] Dono sheets load aur combine
- [x] Audit (nulls, duplicates, cancellations, bad qty/price, countries)
- [x] Cleaning log (step-by-step row counts)
- [x] Non-product codes flag kiye
- [x] RFM table + validation + 5-customer manual check
- [x] EDA (distribution, skew, outliers, correlation, trend, countries, products, one-time customers, Pareto)
- [x] Parquet files save, `src/data_prep.py` bana
- [ ] Har "Insight" cell me apne words me likho
- [ ] Notebook GitHub pe commit karo (`notebooks/01_eda_rfm.ipynb`)
- [ ] README me problem, dataset, cleaning table, Day 1 findings likho

### Kal (Day 2)
`rfm.parquet` load karo -> `log1p` -> `StandardScaler` -> K-Means (elbow + silhouette) -> DBSCAN -> GMM -> final k.